# Smart Money Event Analysis

Congress, senator, and insider behavior analysis from Quant Warehouse target-engineering event artifacts.

This notebook is intentionally output-first for GitHub review. It loads precomputed EDA artifacts from `research/smart_money_eda`, which is gitignored. The committed notebook keeps the rendered outputs so the analysis is visible on GitHub.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

ARTIFACT_DIR = Path("../../research/smart_money_eda")
if not ARTIFACT_DIR.exists():
    ARTIFACT_DIR = Path("research/smart_money_eda")

required = [
    "robust_congress_by_chamber.csv",
    "robust_congress_disclosure_tradeable.csv",
    "robust_congress_actor_min15.csv",
    "robust_insider_reliable_by_type.csv",
    "robust_insider_reliable_by_role.csv",
    "robust_insider_inferred_by_source.csv",
]
missing = [name for name in required if not (ARTIFACT_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing smart-money EDA artifacts under {ARTIFACT_DIR}: {missing}")

def read_csv(name: str) -> pd.DataFrame:
    return pd.read_csv(ARTIFACT_DIR / name)

def pct_cols(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    out = df.copy()
    for col in cols:
        if col in out.columns:
            out[col] = (pd.to_numeric(out[col], errors="coerce") * 100).round(2)
    return out

def show_percent_table(df: pd.DataFrame, percent_cols: list[str], sort_cols: list[str] | None = None, ascending=False) -> pd.DataFrame:
    out = pct_cols(df, percent_cols)
    if sort_cols:
        out = out.sort_values(sort_cols, ascending=ascending)
    return out.reset_index(drop=True)

congress = read_csv("robust_congress_by_chamber.csv")
congress_tradeable = read_csv("robust_congress_disclosure_tradeable.csv")
congress_actors = read_csv("robust_congress_actor_min15.csv")
insider_type = read_csv("robust_insider_reliable_by_type.csv")
insider_role = read_csv("robust_insider_reliable_by_role.csv")
insider_inferred = read_csv("robust_insider_inferred_by_source.csv")

print(f"artifact_dir={ARTIFACT_DIR.resolve()}")
print({
    "congress_rows": len(congress),
    "congress_actor_rows": len(congress_actors),
    "insider_type_rows": len(insider_type),
    "insider_role_rows": len(insider_role),
})

artifact_dir=/home/jlee153232/PycharmProjects/quant-warehouse/research/smart_money_eda
{'congress_rows': 4, 'congress_actor_rows': 62, 'insider_type_rows': 2, 'insider_role_rows': 8}


## Method

The tables use side-adjusted abnormal returns around event dates. For buys, positive means the traded security outperformed SPY after the event. For sells, positive would mean the security underperformed SPY after the event.

`winsor_*` columns are 1st/99th percentile winsorized to reduce the impact of extreme small-cap, split, and data-quality outliers. I rely on those columns more than raw averages.

## Congress: House vs Senate

This is the cleanest part of the current smart-money label set. Congress data has enough actors/events to compare House and Senate, but remember that event-date results are not necessarily tradable because disclosures are delayed.

In [2]:
cols = [
    "chamber", "event_type", "events", "symbols", "actors",
    "winsor_side_20d", "median_side_20d", "hit_side_20d",
    "winsor_side_60d", "median_side_60d", "hit_side_60d",
    "winsor_side_120d", "median_side_120d", "hit_side_120d",
]
show_percent_table(
    congress[cols],
    [c for c in cols if c.startswith(("winsor_", "median_", "hit_"))],
    sort_cols=["event_type", "winsor_side_120d"],
    ascending=[True, False],
)

,chamber,event_type,events,symbols,actors,winsor_side_20d,median_side_20d,hit_side_20d,winsor_side_60d,median_side_60d,hit_side_60d,winsor_side_120d,median_side_120d,hit_side_120d
0,Senate,congress_buy,623,7,36,1.9400,1.2500,60.8300,5.1600,3.4300,59.4400,12.6500,6.9000,61.4200
1,House,congress_buy,1766,7,112,1.4900,1.1100,55.8100,4.1400,1.5200,54.8800,7.1400,2.3400,55.0600
2,House,congress_sell,1626,7,116,-1.3400,-1.0400,43.2100,-4.0700,-1.9600,43.9900,-9.9500,-4.7400,40.1500
3,Senate,congress_sell,565,7,39,-2.0300,-1.3400,40.7100,-5.0100,-2.5700,40.6800,-10.0800,-5.0600,36.3800


### Congress Takeaways

- Senate buys are the strongest chamber/type bucket in this run, especially at 60-120 trading days.
- House buys are also positive, but weaker than Senate buys.
- House and Senate sells are negative side-adjusted signals. That means treating reported sells as bearish would have lost money on average in this sample because the sold names generally kept outperforming.
- The useful target is probably not a naive `buy good / sell bad` label. It is more likely actor-aware, chamber-aware, lag-aware, and maybe direction asymmetric.

## Tradeability: Disclosure Date vs Transaction Date

Congress transactions are often disclosed weeks after the trade. This table measures performance from the disclosure/tradeable date instead of the transaction date.

In [3]:
tradeable_cols = [
    "chamber", "event_type", "events", "disclosure_rate", "median_lag_days", "avg_lag_days",
    "winsor_tradeable_side_20d", "median_tradeable_side_20d", "hit_tradeable_side_20d",
    "winsor_tradeable_side_60d", "median_tradeable_side_60d", "hit_tradeable_side_60d",
    "winsor_tradeable_side_120d", "median_tradeable_side_120d", "hit_tradeable_side_120d",
]
show_percent_table(
    congress_tradeable[tradeable_cols],
    ["disclosure_rate", "winsor_tradeable_side_20d", "median_tradeable_side_20d", "hit_tradeable_side_20d",
     "winsor_tradeable_side_60d", "median_tradeable_side_60d", "hit_tradeable_side_60d",
     "winsor_tradeable_side_120d", "median_tradeable_side_120d", "hit_tradeable_side_120d"],
    sort_cols=["event_type", "winsor_tradeable_side_120d"],
    ascending=[True, False],
)

,chamber,event_type,events,disclosure_rate,median_lag_days,avg_lag_days,winsor_tradeable_side_20d,median_tradeable_side_20d,hit_tradeable_side_20d,winsor_tradeable_side_60d,median_tradeable_side_60d,hit_tradeable_side_60d,winsor_tradeable_side_120d,median_tradeable_side_120d,hit_tradeable_side_120d
0,Senate,congress_buy,623,71.1100,27.0000,91.5169,1.6400,1.2000,59.8400,5.0800,3.1900,61.7800,11.0100,6.8200,59.3900
1,House,congress_buy,1766,82.5600,26.0000,59.1118,1.3600,0.7200,53.7400,3.9300,1.4900,55.5800,6.8800,2.4600,55.3500
2,Senate,congress_sell,565,68.6700,27.0000,79.6392,-1.6900,-1.0500,40.7900,-4.8100,-2.9800,38.8500,-9.1400,-5.1100,36.2900
3,House,congress_sell,1626,87.5200,28.0000,53.9501,-1.3300,-0.9300,43.7500,-4.8700,-1.6700,43.4400,-10.1600,-4.1400,40.7200


### Tradeability Takeaways

- The median disclosure lag is roughly four weeks. Average lag is much worse, especially for Senate buys.
- Senate buys remain positive even from disclosure date, which is more interesting than transaction-date performance.
- Sell signals remain poor from disclosure date. I would not train a simple bearish target from congressional sells without actor-specific controls.

## Actor-Level Congress Patterns

Actor-level summaries are more promising than aggregate labels, but they are also more fragile. These rows require at least 15 events.

In [4]:
actor_cols = [
    "actor_name", "chamber", "event_type", "events", "symbols",
    "winsor_side_20d", "median_side_20d", "hit_side_20d",
    "winsor_side_60d", "median_side_60d", "hit_side_60d",
    "winsor_side_120d", "median_side_120d", "hit_side_120d",
]
show_percent_table(
    congress_actors[actor_cols].query("event_type == 'congress_buy'"),
    [c for c in actor_cols if c.startswith(("winsor_", "median_", "hit_"))],
    sort_cols=["winsor_side_120d"],
    ascending=False,
).head(15)

,actor_name,chamber,event_type,events,symbols,winsor_side_20d,median_side_20d,hit_side_20d,winsor_side_60d,median_side_60d,hit_side_60d,winsor_side_120d,median_side_120d,hit_side_120d
0,Markwayne Mullin,Senate,congress_buy,61,6,2.6400,0.8500,62.3000,3.8600,-1.5800,39.3400,25.8100,10.6000,60.4700
1,Donald Sternoff Beyer,House,congress_buy,17,4,3.4600,2.5200,82.3500,11.1500,12.5800,94.1200,22.9400,24.9800,70.5900
2,Rob Bresnahan,House,congress_buy,27,6,2.5800,1.7000,59.2600,9.3100,7.1900,70.3700,19.9800,17.7900,81.4800
3,Michael Mccaul,House,congress_buy,98,6,1.4000,1.8700,64.2900,9.7200,6.1000,63.5400,19.5100,8.8200,72.3400
4,Pat Roberts,Senate,congress_buy,107,5,4.6900,2.5200,65.4200,7.0600,6.8300,68.2200,15.7700,8.5400,61.6800
5,Shelley Moore Capito,Senate,congress_buy,31,4,0.4800,1.0400,61.2900,5.3500,5.3400,70.9700,14.7500,11.7800,75.8600
6,Jared Moskowitz,House,congress_buy,23,6,0.7200,1.0200,56.5200,13.6100,9.3100,73.9100,14.6800,5.6500,65.2200
7,Katie Britt,Senate,congress_buy,22,5,3.4000,6.0500,63.6400,10.0400,9.2900,63.6400,14.1600,2.2100,72.7300
8,Pete Sessions,House,congress_buy,32,4,1.9600,1.1200,62.5000,7.8700,2.4400,65.6200,13.6200,2.0500,62.5000
9,Tommy Tuberville,Senate,congress_buy,18,6,-3.8300,-3.7700,38.8900,2.6100,1.8600,55.5600,10.3600,6.8100,77.7800


In [5]:
show_percent_table(
    congress_actors[actor_cols].query("event_type == 'congress_sell'"),
    [c for c in actor_cols if c.startswith(("winsor_", "median_", "hit_"))],
    sort_cols=["winsor_side_120d"],
    ascending=False,
).head(15)

,actor_name,chamber,event_type,events,symbols,winsor_side_20d,median_side_20d,hit_side_20d,winsor_side_60d,median_side_60d,hit_side_60d,winsor_side_120d,median_side_120d,hit_side_120d
0,Shelley Moore Capito,Senate,congress_sell,89,4,0.4300,0.5400,53.9300,1.3400,3.6400,59.7700,1.0900,-0.6400,47.4400
1,Kathy Manning,House,congress_sell,30,6,0.0500,2.1300,63.3300,4.1100,3.5700,63.3300,-0.0300,2.1800,53.3300
2,Julie Johnson,House,congress_sell,28,6,2.0600,3.5600,57.1400,0.8800,2.9300,53.5700,-0.8200,-1.9500,50.0000
3,Valerie Hoyle,House,congress_sell,16,7,-0.2400,-0.3800,37.5000,0.6500,0.7000,56.2500,-1.5300,2.6300,62.5000
4,Patrick Fallon,House,congress_sell,31,3,-0.4100,-1.0000,41.9400,1.5300,2.8700,64.5200,-1.6600,-0.7700,48.3900
5,Suzan K. Delbene,House,congress_sell,17,1,-0.8700,-0.0100,41.1800,-3.4700,-2.3700,47.0600,-3.0700,-1.4600,41.1800
6,Julia Letlow,House,congress_sell,15,5,4.2000,3.4000,73.3300,4.6600,2.7300,73.3300,-3.5500,-1.1900,28.5700
7,Lisa Mcclain,House,congress_sell,38,6,-3.6100,-4.0000,23.6800,-4.0700,-0.1100,47.3700,-5.3200,-2.1500,44.7400
8,Ro Khanna,House,congress_sell,281,7,0.6200,0.9600,55.9100,-1.2600,-0.1100,48.7200,-6.3900,-2.4400,43.4900
9,Michael Mccaul,House,congress_sell,239,6,-1.0500,-1.0700,43.5100,-1.0500,-0.1600,49.3700,-7.1500,0.9800,50.6300


### Actor-Level Takeaways

- Some individual buy actors show much stronger forward behavior than the chamber averages.
- The event count threshold is still low; these are research leads, not production labels yet.
- For target engineering, actor identity should be a metadata dimension, not just a free-text column. Good follow-up labels would include actor historical hit rate, actor recency, chamber, disclosure lag, and amount bucket.

## Insider Trades

The reliable insider subset is much smaller because many stored FMP insider rows have sparse/null text fields. This section uses only transaction-type classified rows, which are cleaner but narrow.

In [6]:
insider_cols = [
    "event_type", "events", "symbols", "actors",
    "winsor_side_20d", "median_side_20d", "hit_side_20d",
    "winsor_side_60d", "median_side_60d", "hit_side_60d",
    "winsor_side_120d", "median_side_120d", "hit_side_120d",
]
show_percent_table(
    insider_type[insider_cols],
    [c for c in insider_cols if c.startswith(("winsor_", "median_", "hit_"))],
    sort_cols=["event_type"],
    ascending=True,
)

,event_type,events,symbols,actors,winsor_side_20d,median_side_20d,hit_side_20d,winsor_side_60d,median_side_60d,hit_side_60d,winsor_side_120d,median_side_120d,hit_side_120d
0,insider_buy,38,4,15,4.1600,2.6100,61.1100,3.8900,-1.3000,47.0600,5.8300,12.9800,56.2500
1,insider_sell,723,7,76,-1.6300,-1.6000,42.2200,-2.3000,-0.4300,47.1000,-3.1200,-0.8700,48.1400


In [7]:
role_cols = [
    "role_bucket", "event_type", "events", "symbols", "actors",
    "winsor_side_20d", "median_side_20d", "hit_side_20d",
    "winsor_side_60d", "median_side_60d", "hit_side_60d",
    "winsor_side_120d", "median_side_120d", "hit_side_120d",
]
show_percent_table(
    insider_role[role_cols],
    [c for c in role_cols if c.startswith(("winsor_", "median_", "hit_"))],
    sort_cols=["event_type", "winsor_side_120d"],
    ascending=[True, False],
)

,role_bucket,event_type,events,symbols,actors,winsor_side_20d,median_side_20d,hit_side_20d,winsor_side_60d,median_side_60d,hit_side_60d,winsor_side_120d,median_side_120d,hit_side_120d
0,cfo,insider_buy,5,1,2,20.2700,17.0600,75.0000,26.5200,24.1300,50.0000,31.0900,21.6700,75.0000
1,director,insider_buy,16,3,9,1.4200,1.1700,56.2500,-0.4600,-1.3000,43.7500,9.7600,15.9700,66.6700
2,ceo_president,insider_buy,4,1,1,3.2400,4.8300,66.6700,11.5300,6.7500,66.6700,2.3000,-2.6600,33.3300
3,other_officer,insider_buy,13,2,4,2.7400,3.7400,61.5400,-0.4700,-10.1700,45.4500,-8.6400,-22.4000,40.0000
4,other_officer,insider_sell,256,6,27,-0.0200,0.1400,50.8300,1.5400,2.0800,58.0400,1.6400,2.2500,55.3800
5,ceo_president,insider_sell,259,7,19,-2.0600,-2.0200,37.8000,-4.1200,-1.7400,40.8900,-3.4800,-0.4300,48.1000
6,cfo,insider_sell,60,4,5,-2.5900,-1.7800,45.7600,-2.8300,-0.9100,48.2100,-5.9100,-2.4100,44.4400
7,director,insider_sell,148,7,27,-3.5700,-3.6000,33.8100,-5.3400,-4.7200,39.5300,-9.9400,-4.9500,37.3900


### Insider Takeaways

- Reliable insider buys are positive in this sample, but only 38 events across 4 symbols, so the evidence is thin.
- Insider sells are not useful as a simple bearish target. They are negative side-adjusted, meaning stocks often rose after sells.
- CFO buys look very strong, but the sample is only 5 events. Treat this as a high-priority research lead, not a conclusion.
- CEO/director/officer role buckets should be retained as metadata for labels.

## Inferred Insider Direction Caveat

I also tested a holdings-delta heuristic to infer buy/sell direction when FMP transaction text is missing. It expands coverage dramatically but introduces serious outlier/data-quality problems.

In [8]:
inferred_cols = [
    "classification_source", "event_type", "events", "symbols", "actors",
    "avg_side_120d", "winsor_side_120d", "median_side_120d", "hit_side_120d",
]
show_percent_table(
    insider_inferred[inferred_cols],
    ["avg_side_120d", "winsor_side_120d", "median_side_120d", "hit_side_120d"],
    sort_cols=["classification_source", "event_type"],
    ascending=True,
)

,classification_source,event_type,events,symbols,actors,avg_side_120d,winsor_side_120d,median_side_120d,hit_side_120d
0,holdings_delta,insider_buy,88501,5109,3,415.1100,-1.6300,-4.7500,41.2800
1,holdings_delta,insider_sell,88827,4992,15,-437.8100,4.5200,6.3500,61.1600
2,transaction_type,insider_buy,38,4,15,6.1600,5.8300,12.9800,56.2500
3,transaction_type,insider_sell,591,7,71,-2.8400,-2.6400,0.1000,50.5100


### Inference Takeaways

- Raw means from holdings-delta inferred rows are not trustworthy. The average 120-day values are dominated by extreme outliers.
- Winsorized and median values tell a different story, which means the inferred label needs robust filters before being used.
- I would not add holdings-delta inferred insider labels to production target engineering until we add liquidity/price/split sanity checks and source-quality flags.

## Target Engineering Recommendations

1. Add a `smart_money` event-label family, but keep subfamilies separate: `congress`, `insider`, `institutional`.
2. For Congress, use disclosure-date labels for tradable research. Keep transaction-date labels only as explanatory/non-tradable metadata.
3. For Congress buys, include chamber and actor identity. Senate buys look strongest in this sample.
4. Do not treat Congress sells or insider sells as automatically bearish. Current results suggest that is too naive.
5. For insiders, prefer explicit transaction-type labels first. Treat holdings-delta inference as experimental until cleaned.
6. Store quality metadata with every event label: source, classification method, lag days, actor role/chamber, amount bucket, and coverage flags.